# Adaptive Step Size and Online Calibration

Two halves:

- **Part 1** uses the logistic ODE -- a smooth, easy problem -- to demonstrate **post-hoc calibration**, the relationship between the per-step quasi-MLE and the global MLE, and the equivalence of post-hoc and online schemes on smooth problems.
- **Part 2** uses a **two-component logistic with staggered transitions** ($x_1(0) = 10^{-5}$, $x_2(0) = 10^{-10}$) -- a multi-scale problem -- to compare the four calibration modes and demonstrate **adaptive step control**.

**Defaults note.** `ekf1_sqr_adaptive_loop` defaults to `calibration="dynamic"` (probdiffeq's `MLEDiffusion` -- statistically honest per-step, but **fails on multi-scale ODEs**) and `sigma_in_error="running_mean"` (smooths the controller signal using a running mean of past $\widehat\sigma^2$ values; typically reduces reject counts by 3-7x with no accuracy regression). When the ODE has multiple components on different scales, use `calibration="diagonal"` -- a one-line change with no other cost.


In [ ]:
import jax
import jax.numpy as np
import jax.scipy.stats as jss
import matplotlib.pyplot as plt
import numpy as onp

from ode_filters.calibration import (
    posthoc_mle_sigma_sqr,
    quasi_mle_sigma_sqr,
    rescale_sqr_seq,
)
from ode_filters.filters import (
    ekf1_sqr_adaptive_loop,
    ekf1_sqr_loop,
    ekf1_sqr_loop_dynamic,
)
from ode_filters.measurement import ODEInformation
from ode_filters.priors import IWP, taylor_mode_initialization

jax.config.update('jax_enable_x64', True)

# -----------------------------------------------------------------------------
# Unified plot style: applied once, used by every figure below.
# -----------------------------------------------------------------------------
plt.rcParams.update({
    'figure.dpi': 110,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linewidth': 0.6,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'lines.linewidth': 1.3,
    'lines.markersize': 4,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'legend.fontsize': 9,
    'legend.frameon': False,
})

# Standard figure sizes used throughout the notebook.
FIG_WIDE     = (9.0, 3.4)   # single-axis x(t) plots
FIG_STACK2   = (9.0, 5.4)   # two stacked axes (x_1, x_2)
FIG_SIDE2    = (10.0, 4.0)  # two side-by-side panels
FIG_SQUARE   = (5.6, 5.6)   # phase / Q-Q

# Calibration-mode colour palette. Keyed by mode name so legends match
# across every figure.
MODE = {
    'analytic':      dict(color='black',   marker=None, ls='-',  alpha=0.55),
    'post-hoc':      dict(color='#1f77b4', marker='o',  ls='-'),   # blue
    'online':        dict(color='#17becf', marker='s',  ls='--'),  # teal
    'dynamic':       dict(color='#d62728', marker='o',  ls='-'),   # red    -- fails on multi-scale
    'cumulative':    dict(color='#9467bd', marker='s',  ls='--'),  # purple -- legacy
    'diagonal':      dict(color='#2ca02c', marker='D',  ls='-'),   # green  -- recommended
    'diagonal_ekf0': dict(color='#ff7f0e', marker='v',  ls=':'),   # orange -- alternative
}

def style(mode):
    """Return matplotlib kwargs for a calibration mode."""
    return MODE[mode].copy()


## Part 1 -- Post-hoc calibration on the logistic ODE

$$\dot x = x(1-x), \qquad x(0)=0.1, \qquad t \in [0, 8].$$

We run a fixed-step EKF1 with $\sigma = 1$, then ask: *what scalar* $\sigma$ would have made the recorded residuals look like draws from a well-calibrated filter? Two equivalent answers:

- **Per-step quasi-MLE** (Bosch et al. 2021 Eq. 32): $\widehat\sigma^2_n = m_z^{(n) \top} S_n^{-1} m_z^{(n)} / d$.
- **Post-hoc joint MLE** (closed-form under constant $\sigma$): $\widehat\sigma^2_{\text{MLE}} = (1 / N) \sum_n \widehat\sigma^2_n$.

On a smooth, well-specified problem they agree exactly. Then we rescale every stored covariance by $\sqrt{\widehat\sigma^2_{\text{MLE}}}$.


In [ ]:
def vf_log(x, *, t):
    return x * (1 - x)


x0_log = np.array([0.1])
tspan_log = (0.0, 8.0)
prior_log = IWP(q=2, d=1)
mu0_log, S0_log = taylor_mode_initialization(vf_log, x0_log, q=2)
measure_log = ODEInformation(vf_log, prior_log.E0, prior_log.E1)

N_log = 80
res_log = ekf1_sqr_loop(mu0_log, S0_log, prior_log, measure_log, tspan_log, N_log)
m_log = np.stack(list(res_log[0]))
P_sqr_log = np.stack(list(res_log[1]))
mz_log = np.stack(list(res_log[-3]))
Pz_sqr_log = np.stack(list(res_log[-2]))
ts_log = onp.linspace(tspan_log[0], tspan_log[1], N_log + 1)

# Per-step quasi-MLE (one number per step).
sigma_sqr_per_step = onp.asarray(
    [float(quasi_mle_sigma_sqr(mz_log[i], Pz_sqr_log[i])) for i in range(N_log)]
)
# Post-hoc joint MLE.
sigma_sqr_post = float(posthoc_mle_sigma_sqr(mz_log, Pz_sqr_log))

print(f'per-step quasi-MLE: mean = {sigma_sqr_per_step.mean():.6g}')
print(f'post-hoc joint MLE:        {sigma_sqr_post:.6g}')
print(f'absolute difference:       {abs(sigma_sqr_per_step.mean() - sigma_sqr_post):.2e}')


The two numbers agree to machine precision. Now rescale the stored covariances by $\sqrt{\widehat\sigma^2_{\text{MLE}}}$ and visualise the before/after.


In [ ]:
P_sqr_log_cal = rescale_sqr_seq(P_sqr_log, sigma_sqr_post)
P_log = np.einsum('nij,nik->njk', P_sqr_log, P_sqr_log)
P_log_cal = np.einsum('nij,nik->njk', P_sqr_log_cal, P_sqr_log_cal)
std_uncal = onp.sqrt(onp.asarray(P_log[:, 0, 0]))
std_cal = onp.sqrt(onp.asarray(P_log_cal[:, 0, 0]))
x_log = onp.asarray(m_log[:, 0])

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=FIG_STACK2, sharex=True)
ax1.plot(ts_log, x_log, color='black', lw=1.2, label='filtered mean')
ax1.fill_between(
    ts_log, x_log - 2 * std_cal, x_log + 2 * std_cal,
    color=MODE['post-hoc']['color'], alpha=0.4,
    label=r'post-hoc 2$\sigma$ (calibrated)',
)
ax1.set_ylabel('x(t)')
ax1.set_ylim(0.0, 1.1)
ax1.set_title('Logistic ODE: filtered mean + post-hoc 2$\\sigma$ band')
ax1.legend(loc='lower right')

ax2.semilogy(
    ts_log, std_uncal, color='grey', lw=1.2,
    label=r'uncalibrated $\sigma_x(t)$  ($\sigma=1$ default)',
)
ax2.semilogy(
    ts_log, std_cal, color=MODE['post-hoc']['color'], lw=1.4,
    label=r'post-hoc calibrated $\sigma_x(t)$',
)
ax2.set_xlabel('t')
ax2.set_ylabel(r'$\sigma_x(t)$  (log)')
ax2.set_title(
    rf'Calibration shrinks posterior std uniformly by '
    rf'$\sqrt{{\widehat\sigma^2}} \approx {sigma_sqr_post**0.5:.2g}$'
)
ax2.legend(loc='lower right')
plt.tight_layout()
plt.show()


Post-hoc calibration scales every posterior standard deviation by $\sqrt{\widehat\sigma^2}$ -- here a strong shrinkage, because the IWP$(2)$ prior is loose for this smooth problem. The *shape* of the $\sigma_x(t)$ curve is unchanged: post-hoc calibration is one scalar applied to every covariance.


## Part 1b -- Online dynamic: same order-of-magnitude, different shape

`ekf1_sqr_loop_dynamic` is the *online* counterpart to post-hoc: per-step $\widehat\sigma^2_n$ is baked into the current step's $Q_h$ *before* propagation (probdiffeq's `MLEDiffusion`). On a smooth single-component problem the per-step $\widehat\sigma^2$ values are *approximately* constant, so online and post-hoc agree on the *typical scale* of the posterior $\sigma_x(t)$. They differ in *shape*: online bakes each step's scale into the propagation (so past steps keep their own scaling), while post-hoc multiplies every stored covariance by one global $\widehat\sigma^2_{\text{MLE}}$ at the end. The result is similar in magnitude but with a different decay profile.


In [ ]:
res_dyn_log = ekf1_sqr_loop_dynamic(
    mu0_log, S0_log, prior_log, measure_log, tspan_log, N=N_log
)
m_dyn_log = onp.stack([onp.asarray(m) for m in res_dyn_log[0]])
P_dyn_log = onp.stack(
    [onp.asarray(P_sqr.T @ P_sqr) for P_sqr in res_dyn_log[1]]
)
std_dyn_log = onp.sqrt(P_dyn_log[:, 0, 0])
ts_dyn_log = onp.linspace(tspan_log[0], tspan_log[1], len(res_dyn_log[0]))
sigma_dyn_log = onp.asarray(res_dyn_log[9])

print(
    f'global post-hoc sigma^2 = {sigma_sqr_post:.3e}  |  '
    f'online per-step mean = {sigma_dyn_log.mean():.3e}  |  '
    f'per-step spread max/min = {sigma_dyn_log.max() / sigma_dyn_log.min():.2g}'
)

fig, ax = plt.subplots(figsize=FIG_WIDE)
ax.semilogy(
    ts_log, std_cal, color=MODE['post-hoc']['color'], lw=1.4,
    label=r'post-hoc $\sigma_x(t)$',
)
ax.semilogy(
    ts_dyn_log, std_dyn_log, color=MODE['online']['color'], ls='--', lw=1.4,
    label=r'online dynamic $\sigma_x(t)$',
)
ax.set_xlabel('t')
ax.set_ylabel(r'$\sigma_x(t)$  (log)')
ax.set_title(
    'Online dynamic vs post-hoc on a smooth problem: same scale, '
    'different shape'
)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


## Part 2 -- Adaptive control on a multi-scale staggered problem

$$\dot x_i = r\, x_i (1 - x_i), \qquad r = 2, \qquad x_1(0) = 10^{-5}, \qquad x_2(0) = 10^{-10}, \qquad t \in [0, 15].$$

$x_1$ transitions around $t \approx 5.8$ and $x_2$ around $t \approx 11.5$ -- two well-separated fast regions with the components living **5 orders of magnitude apart**. The closed-form solution $x_i(t) = 1 / (1 + (1/x_i(0) - 1)e^{-rt})$ provides a reference.

We run **all four calibration modes** with shared kwargs and show first the headline contrast: the default `"dynamic"` *fails* on $x_2$ because the scalar $\widehat\sigma^2$ is dominated by $x_1$ and collapses once $x_1$ is well-tracked; the recommended `"diagonal"` *resolves* it via per-component $\widehat\sigma^2_i$.


In [ ]:
R_RATE = 2.0
X0_VEC = onp.array([1e-5, 1e-10])


def vf_dl(x, *, t):
    return np.array(
        [R_RATE * x[0] * (1 - x[0]), R_RATE * x[1] * (1 - x[1])]
    )


def analytic(t, x0, r):
    return 1.0 / (1.0 + (1.0 / x0 - 1.0) * onp.exp(-r * t))


tspan_dl = (0.0, 15.0)
q_dl = 3
prior_dl = IWP(q=q_dl, d=2)
mu0_dl, S0_dl = taylor_mode_initialization(vf_dl, np.asarray(X0_VEC), q=q_dl)
measure_dl = ODEInformation(vf_dl, prior_dl.E0, prior_dl.E1)

# Common tolerances. `sigma_in_error="running_mean"` is the default; passed
# explicitly for clarity.
common_kwargs = dict(
    atol=1e-5, rtol=1e-3, h_min=1e-9, sigma_in_error='running_mean',
)
runs = {
    mode: ekf1_sqr_adaptive_loop(
        mu0_dl, S0_dl, prior_dl, measure_dl, tspan_dl,
        calibration=mode, **common_kwargs,
    )
    for mode in ('dynamic', 'cumulative', 'diagonal', 'diagonal_ekf0')
}
for mode, r in runs.items():
    h_lo, h_hi = min(r.h_seq), max(r.h_seq)
    print(
        f'{mode:14s}: accepted = {len(r.h_seq):3d}, rejected = {r.n_rejected:3d}, '
        f'h in [{h_lo:.3g}, {h_hi:.3g}]'
    )


In [ ]:
def _components(r):
    ts = onp.asarray(r.t_seq)
    m = onp.stack([onp.asarray(mi) for mi in r.m_seq])
    x = m @ onp.asarray(prior_dl.E0.T)
    P_sqr = onp.stack([onp.asarray(P) for P in r.P_seq_sqr])
    P = onp.einsum('nij,nik->njk', P_sqr, P_sqr)
    cov_x = onp.einsum(
        'ij,njk,lk->nil', onp.asarray(prior_dl.E0), P,
        onp.asarray(prior_dl.E0)
    )
    return ts, x, cov_x[:, 0, 0], cov_x[:, 1, 1]

ts_dyn, x_dyn, _, var_dyn_2 = _components(runs['dynamic'])
ts_dia, x_dia, _, var_dia_2 = _components(runs['diagonal'])

ts_dense = onp.linspace(tspan_dl[0], tspan_dl[1], 600)
x_true_0 = analytic(ts_dense, float(X0_VEC[0]), R_RATE)
x_true_1 = analytic(ts_dense, float(X0_VEC[1]), R_RATE)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=FIG_STACK2, sharex=True)
for ax, idx, comp, x_true, vdia in [
    (ax1, 0, 'x_1', x_true_0, None),
    (ax2, 1, 'x_2', x_true_1, var_dia_2),
]:
    ax.plot(ts_dense, x_true, **MODE['analytic'], label='analytic')
    sty = style('dynamic')
    ax.plot(
        ts_dyn, x_dyn[:, idx], **sty,
        label=f"dynamic (default, {len(runs['dynamic'].h_seq)} steps)",
    )
    sty = style('diagonal')
    ax.plot(
        ts_dia, x_dia[:, idx], **sty,
        label=f"diagonal (recommended, {len(runs['diagonal'].h_seq)} steps)",
    )
    if vdia is not None:
        ax.fill_between(
            ts_dia, x_dia[:, idx] - 2 * onp.sqrt(vdia),
            x_dia[:, idx] + 2 * onp.sqrt(vdia),
            color=MODE['diagonal']['color'], alpha=0.20,
        )
    ax.set_ylabel(f'${comp}(t)$')
    ax.set_ylim(-0.1, 1.2)
ax2.set_xlabel('t')
ax1.set_title(
    'Multi-scale staggered logistic: dynamic misses $x_2$, diagonal resolves it'
)
ax1.legend(loc='center right')
plt.tight_layout()
plt.show()


On $x_1$ both modes agree closely with the analytic solution. On $x_2$ they diverge: **dynamic stays near zero** -- the filter never moves $x_2$ -- because once $x_1$ is well-tracked, the scalar $\widehat\sigma^2$ (averaged across components) collapses to a tiny value, and the prior process noise gets scaled down with it, starving $x_2$ of the slack it needs to grow. **Diagonal** estimates $\widehat\sigma^2_i$ separately per component, so $x_2$ keeps its own noise budget. Both modes use exactly the same tolerances and same step controller; only `calibration=` differs.


### Full comparison: all four modes + per-component diffusion

For the curious: the same multi-scale problem under all four modes. `cumulative` is the legacy multiplicative scheme (resolves $x_2$ by inheriting an inflated covariance from $x_1$'s transition -- non-Markovian but works). `diagonal_ekf0` uses $H_0 = E_1$ so the denominator $E_1 Q E_1^\top$ is *exactly* diagonal; behaves very similarly to `diagonal` (which uses the EKF1 Jacobian) and diverges from it only during the fast transitions where the linearisation matters most.


In [ ]:
x_true_endpoint = onp.array([
    analytic(tspan_dl[1], float(X0_VEC[0]), R_RATE),
    analytic(tspan_dl[1], float(X0_VEC[1]), R_RATE),
])

row_labels = {
    'dynamic':       'dynamic        (scalar, probdiffeq, default)',
    'cumulative':    'cumulative     (legacy, scalar)',
    'diagonal':      'diagonal       (recommended; EKF1 denom)',
    'diagonal_ekf0': 'diagonal_ekf0  (EKF0 denom, exact diagonal)',
}
print(f'{"mode":48s} {"acc":>4s} {"rej":>4s}  '
      f'{"err_x1":>10s} {"err_x2":>10s}')
print('-' * 82)
for mode, label in row_labels.items():
    r = runs[mode]
    m_final = onp.asarray(r.m_seq[-1])
    x_final = m_final @ onp.asarray(prior_dl.E0.T)
    err = onp.abs(x_final - x_true_endpoint)
    print(
        f'{label:48s} {len(r.h_seq):4d} {r.n_rejected:4d}  '
        f'{err[0]:10.2e} {err[1]:10.2e}'
    )

# Per-component sigma traces from the recommended diagonal mode -- the
# diagnostic that justifies the diagonal modes. sigma_1^2 peaks during
# x_1's transition (t ~ 6); sigma_2^2 peaks during x_2's transition
# (t ~ 11) and lives several orders of magnitude below sigma_1^2 in
# between. (The diagonal_ekf0 variant produces nearly identical traces;
# we omit it from the plot for clarity.)
sigma_dia = onp.stack([onp.asarray(s) for s in runs['diagonal'].sigma_sqr_seq])
ts_dia_steps = onp.asarray(runs['diagonal'].t_seq)[1:]

fig, ax = plt.subplots(figsize=FIG_WIDE)
ax.semilogy(
    ts_dia_steps, sigma_dia[:, 0],
    color=MODE['diagonal']['color'], marker='o', ms=4.0, lw=1.0,
    label=r'$\widehat\sigma_1^2$',
)
ax.semilogy(
    ts_dia_steps, sigma_dia[:, 1],
    color=MODE['diagonal_ekf0']['color'], marker='s', ms=4.0, lw=1.0,
    label=r'$\widehat\sigma_2^2$',
)
ax.axvspan(5.0, 7.0, color='lightgrey', alpha=0.25, label='$x_1$ transition')
ax.axvspan(10.5, 12.5, color='lightblue', alpha=0.25, label='$x_2$ transition')
ax.set_xlabel('t')
ax.set_ylabel(r'$\widehat\sigma_i^2$')
ax.set_title(
    r'Per-component diffusion estimates (diagonal mode): '
    r'$\widehat\sigma_1^2$ and $\widehat\sigma_2^2$ peak at their '
    r'respective transitions'
)
ax.legend(loc='lower right', ncol=2)
plt.tight_layout()
plt.show()


**When to pick which mode:**

| Mode | x_2 error | Honest? | Use when |
|---|---|---|---|
| `dynamic` (default) | **fails** | per-step | Single-component or single-scale ODEs |
| **`diagonal`** | **best** | per-step *and* per-component | **Recommended for multi-component ODEs with component-scale gaps** |
| `diagonal_ekf0` | similar to `diagonal` | per-step *and* per-component | Mathematically clean alternative (exact-diagonal denominator) |
| `cumulative` | resolves | non-Markovian | Non-diagonal $\Xi$, or legacy reproducibility |
| `none` | n/a | -- | Diagnostics + post-hoc `posthoc_mle_sigma_sqr` / `rescale_sqr_seq` |

**Defaults.** `sigma_in_error="running_mean"` is the default; it smooths the local-error estimate using a running mean of past $\widehat\sigma^2$ values. The per-step $\widehat\sigma^2$ is still used for the actual $Q$ calibration, so the *posterior* is unchanged -- only step-size decisions are smoother. Pass `sigma_in_error="per_step"` to recover the original Bosch et al. 2021 recipe.

**Multi-d but single-scale.** For ODE systems with $d > 1$ but all components on the *same* scale (e.g. van der Pol), `diagonal` adds estimator noise without resolving anything, and scalar `dynamic` is strictly better. Use the diagonal modes when there is a genuine component-scale gap.


### Step-size trace for the recommended mode

$h(t)$ for the `diagonal` run: small steps cluster at the two transitions, long strides on the plateaus in between. The `sigma_in_error="running_mean"` default keeps the trace smooth -- the chi-squared noise floor of the per-step quasi-MLE no longer dominates step-size decisions.


In [ ]:
r_rec = runs['diagonal']
t_step = onp.asarray(r_rec.t_seq)[1:]
h_seq = onp.asarray(r_rec.h_seq)

fig, ax = plt.subplots(figsize=FIG_WIDE)
ax.semilogy(
    t_step, h_seq, marker='o', color=MODE['diagonal']['color'],
    ms=3.5, lw=0.6,
)
ax.set_xlabel('t')
ax.set_ylabel('$h$  (log)')
ax.set_title(
    f"Adaptive step size: "
    f"{len(h_seq)} accepted, {r_rec.n_rejected} rejected, "
    f"$h$ range {h_seq.min():.2g}-{h_seq.max():.2g}"
)
plt.tight_layout()
plt.show()


Diagonal mode is efficient enough that $h$ stays in a fairly narrow band for this problem: most steps are around $0.3$, with a brief early ramp-up as the controller calibrates. The `sigma_in_error="running_mean"` default keeps the trace smooth -- the $\chi^2$ noise floor of the per-step quasi-MLE no longer dominates step-size decisions, and the reject count stays in single digits.

---

**TL;DR.** On multi-component ODEs with component-scale gaps, use `calibration="diagonal"`. The default `"dynamic"` is fine for single-component / single-scale problems but fails on multi-scale, because its scalar $\widehat\sigma^2$ collapses once the dominant component is well-tracked. The new default `sigma_in_error="running_mean"` damps the chi-squared noise floor on the step controller and is a free improvement on top of any mode.
